# 03 — All-Lecture Agent MVP Using the Reranked RAG Capability

This notebook adds the **agent layer** on top of the shared Lectures 1–34 RAG system created in Notebook 2.

## Architecture

```text
Notebook 2
Lectures 1–34 → shared Chroma → retrieval → reranking
                              │
                              ▼
Notebook 3
Tutor Agent + memory
├─ search_lecture
├─ create_quiz
├─ grade_quiz_answer
└─ get_weak_topics
```

The agent can search the whole course or a specific lecture, while preserving lecture ID, title, timestamp, and video link.


# Step 0: Setup


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q     langchain     langchain-core     langchain-chroma     langchain-huggingface     langchain-openai     langgraph     sentence-transformers

In [ ]:
from pathlib import Path
import json
import os
import operator
import re
from typing import Annotated, Literal
from typing_extensions import NotRequired

PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Engineering_Final_Project")

RAG_CONFIG_PATH = (
    PROJECT_ROOT
    / "rag"
    / "all_lectures_rag_config.json"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEAK_TOPICS_EXPORT = (
    OUTPUT_DIR
    / "all_lecture_weak_topics.json"
)

assert RAG_CONFIG_PATH.exists(), (
    "Run the all-lecture Notebook 2 first. Missing: "
    f"{RAG_CONFIG_PATH}"
)

with open(RAG_CONFIG_PATH, "r", encoding="utf-8") as f:
    rag_config = json.load(f)

print("RAG configuration loaded")
print("Lectures:", rag_config["lecture_numbers"])
print("Collection:", rag_config["collection_name"])

rag_config


# Step 1: Load Notebook 2's Persistent RAG Resources


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import CrossEncoder

embedding_model = HuggingFaceEmbeddings(
    model_name=rag_config["embedding_model"]
)

vectorstore = Chroma(
    collection_name=rag_config["collection_name"],
    embedding_function=embedding_model,
    persist_directory=rag_config["persist_directory"],
)

reranker = CrossEncoder(
    rag_config["reranker_model"]
)

RETRIEVAL_K = rag_config["retrieval_k"]
RERANK_TOP_N = rag_config["rerank_top_n"]

print("Persistent RAG resources loaded")
print("Collection:", rag_config["collection_name"])
print("Retrieve:", RETRIEVAL_K)
print("Rerank top N:", RERANK_TOP_N)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Persistent RAG resources loaded
Collection: mit_18_06_lectures_01_34
Retrieve: 8
Rerank top N: 3


In [ ]:
def seconds_to_timestamp(seconds):
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60

    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    return f"{minutes:02d}:{secs:02d}"


def build_timestamp_link(video_url, seconds):
    return f"{video_url}#t={int(seconds)}"


def retrieve_and_rerank(
    query,
    lecture_number=None,
    candidate_k=RETRIEVAL_K,
    top_n=RERANK_TOP_N,
):
    search_kwargs = {
        "k": candidate_k,
    }

    # Optional lecture-specific search.
    if lecture_number is not None:
        search_kwargs["filter"] = {
            "lecture_number": int(lecture_number)
        }

    candidates = vectorstore.similarity_search(
        query,
        **search_kwargs,
    )

    if not candidates:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )

    final_docs = []

    for rank, (doc, score) in enumerate(
        ranked[:top_n],
        start=1,
    ):
        doc.metadata["rerank_score"] = float(score)
        doc.metadata["rerank_position"] = rank
        final_docs.append(doc)

    return final_docs


Notebook 3 still defines the small `retrieve_and_rerank()` helper because reranking happens **at query time**. However, it does not rebuild chunks, embeddings, or the Chroma collection.


# Step 2: Configure the LLM


- Notebook 2: LLM → turns evidence into answer
- Notebook 3: LLM → also decides which tool to use


In [ ]:
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY loaded")
except Exception:
    print("Add OPENAI_API_KEY to Colab Secrets.")

OPENAI_API_KEY loaded


In [ ]:
from langchain_openai import ChatOpenAI

LLM_MODEL = "gpt-5.6-luna" #changed from "gpt-4o-mini"

llm = ChatOpenAI(
    model=LLM_MODEL,
    #temperature=0,
    reasoning_effort="xhigh",
    use_responses_api=True,
)

print("LLM ready:", LLM_MODEL)

LLM ready: gpt-5.6-luna


# Step 3: Agent State


In [ ]:
from langchain.agents import AgentState


class TutorState(AgentState):
    # Each weak-topic record can keep lecture/source metadata.
    weak_topics: NotRequired[
        Annotated[list[dict], operator.add]
    ]

    current_quiz: NotRequired[dict | None]


# Step 4: Agent Tools


## Step 4.1: Reranked Lecture Search Tool


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command


@tool
def search_lecture(
    query: str,
    lecture_number: int | None = None,
) -> str:
    """
    Search MIT 18.06 Lectures 1–34 using Notebook 2's
    shared vector store and reranker.

    Use lecture_number when the student explicitly asks
    about a specific lecture. Otherwise search all 34 lectures.
    """

    docs = retrieve_and_rerank(
        query=query,
        lecture_number=lecture_number,
    )

    if not docs:
        scope = (
            f"Lecture {lecture_number}"
            if lecture_number is not None
            else "Lectures 1–34"
        )
        return f"No relevant evidence was found in {scope}."

    parts = []

    for i, doc in enumerate(docs, 1):
        start = doc.metadata["start_seconds"]
        end = doc.metadata["end_seconds"]

        parts.append(
            f"SOURCE {i}\n"
            f"LECTURE_ID: {doc.metadata['lecture_id']}\n"
            f"LECTURE_NUMBER: {doc.metadata['lecture_number']}\n"
            f"LECTURE_TITLE: {doc.metadata['lecture_title']}\n"
            f"TIMESTAMP: "
            f"{seconds_to_timestamp(start)}-"
            f"{seconds_to_timestamp(end)}\n"
            f"VIDEO_URL: {doc.metadata['video_url']}\n"
            f"WATCH: "
            f"{build_timestamp_link(doc.metadata['video_url'], start)}\n"
            f"EVIDENCE:\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(parts)


## Step 4.2: Multiple-Choice Quiz Tool


In [ ]:
from pydantic import BaseModel, Field


class QuizItem(BaseModel):
    question: str = Field(
        description="One multiple-choice question"
    )

    options: list[str] = Field(
        min_length=4,
        max_length=4,
        description="Exactly four options without A/B/C/D prefixes"
    )

    correct_answer: Literal[
        "A", "B", "C", "D"
    ]

    topic: str

    explanation: str = Field(
        description="Why the correct answer is correct"
    )

    source_timestamp: str


quiz_llm = llm.with_structured_output(
    QuizItem
)

In [ ]:
@tool
def create_quiz(
    topic: str,
    runtime: ToolRuntime,
    lecture_number: int | None = None,
) -> Command:
    """
    Create one grounded multiple-choice question using
    reranked evidence from Lectures 1–34.

    Use lecture_number when a quiz for a specific lecture
    is requested. Otherwise search across all 34 lectures.
    """

    docs = retrieve_and_rerank(
        query=topic,
        lecture_number=lecture_number,
    )

    if not docs:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content=(
                            "I could not find enough course evidence "
                            "to create a quiz."
                        ),
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    context = "\n\n".join(
        (
            f"Lecture {d.metadata['lecture_number']} — "
            f"{d.metadata['lecture_title']} | "
            f"{seconds_to_timestamp(d.metadata['start_seconds'])}-"
            f"{seconds_to_timestamp(d.metadata['end_seconds'])}: "
            f"{d.page_content}"
        )
        for d in docs
    )

    quiz = quiz_llm.invoke(
        f"""
Create ONE multiple-choice question using only
this reranked MIT 18.06 course evidence.

Topic:
{topic}

Evidence:
{context}

Requirements:
- exactly four options;
- exactly one correct answer;
- options contain text only;
- plausible Linear Algebra distractors;
- do not reveal the answer in the question;
- explanation must say why the correct answer is correct.
"""
    )

    # Source metadata comes from retrieved evidence,
    # not from the LLM.
    primary = docs[0]
    start = primary.metadata["start_seconds"]
    end = primary.metadata["end_seconds"]

    quiz_state = quiz.model_dump()
    quiz_state.update({
        "lecture_id": primary.metadata["lecture_id"],
        "lecture_number": primary.metadata["lecture_number"],
        "lecture_title": primary.metadata["lecture_title"],
        "video_url": primary.metadata["video_url"],
        "timestamp": (
            f"{seconds_to_timestamp(start)}-"
            f"{seconds_to_timestamp(end)}"
        ),
        "video_link": build_timestamp_link(
            primary.metadata["video_url"],
            start,
        ),
    })

    option_lines = "\n".join(
        f"{letter}. {option}"
        for letter, option in zip(
            ["A", "B", "C", "D"],
            quiz.options,
        )
    )

    return Command(
        update={
            "current_quiz": quiz_state,
            "messages": [
                ToolMessage(
                    content=(
                        f"{quiz.question}\n\n"
                        f"{option_lines}\n\n"
                        "Answer with A, B, C, or D."
                    ),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


## Step 4.3: Deterministic Quiz Grading


In [ ]:
@tool
def grade_quiz_answer(
    student_answer: str,
    runtime: ToolRuntime
) -> Command:
    """
    Grade the active A/B/C/D quiz deterministically.

    Wrong answers add the topic plus lecture/source metadata
    to weak_topics.
    """

    quiz = runtime.state.get("current_quiz")

    if not quiz:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="There is no active quiz to grade.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    match = re.match(
        r"\s*([ABCDabcd])",
        student_answer,
    )

    if not match:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="Please answer with A, B, C, or D.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    selected = match.group(1).upper()
    correct = quiz["correct_answer"].upper()

    correct_index = [
        "A", "B", "C", "D"
    ].index(correct)

    correct_text = quiz["options"][correct_index]

    updates = {
        "current_quiz": None
    }

    if selected == correct:
        feedback = (
            "You're right.\n\n"
            f"That's because {quiz['explanation']}"
        )
    else:
        weak_record = {
            "topic": quiz["topic"],
            "lecture_id": quiz.get("lecture_id"),
            "lecture_number": quiz.get("lecture_number"),
            "lecture_title": quiz.get("lecture_title"),
            "timestamp": quiz.get("timestamp"),
            "video_url": quiz.get("video_url"),
            "video_link": quiz.get("video_link"),
        }

        updates["weak_topics"] = [weak_record]

        feedback = (
            "You're wrong.\n\n"
            f"That's because {quiz['explanation']}\n\n"
            f"The correct answer is "
            f"{correct}. {correct_text}."
        )

    updates["messages"] = [
        ToolMessage(
            content=feedback,
            tool_call_id=runtime.tool_call_id,
        )
    ]

    return Command(update=updates)


## Step 4.4: Weak-Topic Tool


In [ ]:
@tool
def get_weak_topics(
    runtime: ToolRuntime
) -> str:
    """Return weak topics for the current student thread."""

    topics = runtime.state.get(
        "weak_topics",
        []
    )

    if not topics:
        return "No weak topics recorded yet."

    # Deduplicate using topic + lecture.
    unique = {}
    for item in topics:
        key = (
            item.get("topic"),
            item.get("lecture_id"),
        )
        unique[key] = item

    lines = ["Topics to review:"]

    for item in unique.values():
        line = (
            f"- {item['topic']} | "
            f"Lecture {item.get('lecture_number')}: "
            f"{item.get('lecture_title')} | "
            f"{item.get('timestamp')}"
        )

        if item.get("video_link"):
            line += f" | {item['video_link']}"

        lines.append(line)

    return "\n".join(lines)


# Step 5: Build Tutor Agent + InMemorySaver


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

checkpointer = InMemorySaver()

SYSTEM_PROMPT = """
You are a Tutor Agent for MIT 18.06 Linear Algebra Lectures 1–34.

CONVERSATION MEMORY

1. Use the full conversation history in the current thread.

2. Resolve references such as "that", "the two", "both",
   "the previous one", and "compare them" using earlier turns.

3. When a follow-up depends on earlier concepts, call search_lecture
   with a standalone query that names those concepts.

OUT OF SCOPE

4. If the student's question is clearly unrelated to
   MIT 18.06 Linear Algebra or the available Lectures 1–34,
   DO NOT answer from general knowledge.

5. Do not call search_lecture for clearly unrelated questions.

6. Respond briefly:
   "That question is outside the available MIT 18.06 Linear Algebra course material."

COURSE ANSWERS

7. For any factual or conceptual course question,
   ALWAYS call search_lecture first.

8. If the student explicitly names a lecture number,
   pass that lecture_number to search_lecture.

9. Otherwise search across all available Lectures 1–34.

10. Base course answers only on evidence returned by search_lecture.

11. Return clean Markdown:
   - short headings when useful;
   - bullet points when useful;
   - concise student-friendly prose.
   - no decorative emphasis or unnecessary literal asterisks.

12. End grounded answers with a Source section containing:
   - lecture number/title;
   - timestamp;
   - clickable video link.

13. Preserve WATCH URLs exactly and render:
    [Watch this part](<WATCH URL>)

QUIZZES

14. If the student asks for a quiz,
    ALWAYS call create_quiz.

15. If the student asks for a quiz from a specific lecture,
    pass that lecture_number to create_quiz.

16. Show one four-option multiple-choice question:
    A. ...
    B. ...
    C. ...
    D. ...

17. Do not reveal the correct answer before the student responds.

18. If an active quiz exists and the student answers it,
    ALWAYS call grade_quiz_answer.

19. Do not grade the quiz yourself.

20. Preserve grading feedback:
    - "You're right." followed by "That's because ..."
    - or "You're wrong." followed by "That's because ..."
      and then the correct answer.

WEAK TOPICS

21. Wrong answers are automatically recorded as weak topics
    with lecture/source metadata.

22. If the student asks what they are struggling with,
    ALWAYS call get_weak_topics.

23. If the student asks for a topic summary,
    ALWAYS call search_lecture first.

RESPONSE FORMATTING

- Use clear Markdown formatting.
- Prefer short sections and bullet points when explaining multiple ideas.
- For definitions, use this structure when appropriate:
  1. short definition
  2. key interpretation or intuition
  3. mathematical expression if useful
  4. source

- Do not output raw LaTeX commands as ordinary text.

- Write inline mathematical expressions between single dollar signs.
  Example:
  $Ax=b$

- Write larger mathematical expressions between double dollar signs.
  Example:
  $$
  \operatorname{Col}(A)
  =
  \operatorname{span}\{\mathbf{a}_1,\mathbf{a}_2,\ldots,\mathbf{a}_n\}
  $$

- Keep mathematical notation simple and compatible with Markdown/MathJax.

- Do not use complicated LaTeX environments such as
  \begin{array}, \begin{matrix}, or custom formatting unless necessary.

- When listing sources, use bullet points.

For conceptual questions, prefer concise student-friendly answers.

Example style:

### Column space

The **column space** of a matrix is the set of all linear combinations
of its columns.

If

$$
A = [\mathbf{a}_1\ \mathbf{a}_2\ \cdots\ \mathbf{a}_n],
$$

then

$$
\operatorname{Col}(A)
=
\operatorname{span}
\{\mathbf{a}_1,\mathbf{a}_2,\ldots,\mathbf{a}_n\}.
$$

Key points:
- It is a subspace.
- It contains every vector that can be produced as $Ax$.
- Some columns can be redundant if they are linear combinations of others.

### Source
- Lecture X — title, timestamp
- Watch this part: URL
"""

agent = create_agent(
    model=llm,
    tools=[
        search_lecture,
        create_quiz,
        grade_quiz_answer,
        get_weak_topics,
    ],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
    state_schema=TutorState,
)

print("Multi-lecture Tutor Agent ready")


Multi-lecture Tutor Agent ready


<>:112: SyntaxWarning: invalid escape sequence '\o'
<>:112: SyntaxWarning: invalid escape sequence '\o'
/tmp/ipykernel_5067/341931383.py:112: SyntaxWarning: invalid escape sequence '\o'
  \operatorname{Col}(A)


# Step 6: Markdown Chat Helper


In [ ]:
from IPython.display import display, Markdown

# Use a fresh thread for the final 34-lecture agent
THREAD_ID = "lectures-01-34-final-agent-test"

config = {
    "configurable": {
        "thread_id": THREAD_ID
    }
}


def extract_text(content):
    """
    Convert either old string responses or
    Responses API structured content into plain text.
    """

    # Old Chat Completions format
    if isinstance(content, str):
        return content

    # Responses API format
    if isinstance(content, list):
        text_parts = []

        for item in content:
            if not isinstance(item, dict):
                continue

            item_type = item.get("type")

            if item_type in {
                "text",
                "output_text",
            }:
                text = item.get("text")

                if isinstance(text, str) and text.strip():
                    text_parts.append(text)

        return "\n\n".join(text_parts)

    return str(content)


def chat(message: str, debug=False):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message,
                }
            ]
        },
        config,
    )

    if debug:
        print("\nTOOL CALLS")

        for msg in result["messages"]:
            if (
                hasattr(msg, "tool_calls")
                and msg.tool_calls
            ):
                print(
                    [
                        call["name"]
                        for call in msg.tool_calls
                    ]
                )

    final_message = result["messages"][-1]

    answer_text = extract_text(
        final_message.content
    )

    # Helpful diagnostic in case no text was extracted
    if not answer_text.strip():
        print(
            "No displayable text found in final response."
        )

        if debug:
            print(
                "\nRAW FINAL CONTENT:"
            )
            print(
                final_message.content
            )

        return result

    display(
        Markdown(answer_text)
    )

    return result


print(
    "Chat helper ready."
)
print(
    "Thread ID:",
    THREAD_ID
)

Chat helper ready.
Thread ID: lectures-01-34-final-agent-test


# Step 7: Interactive Demo


In [ ]:
print("Lectures 1–34 Tutor Agent")
print("Type 'quit' to stop.\n")

while True:
    user_text = input("Student: ").strip()

    if user_text.lower() in {
        "quit",
        "exit",
        "q",
    }:
        print("\nDemo ended.")
        break

    if not user_text:
        continue

    print("\nTutor:")
    chat(user_text)
    print()


Lectures 1–34 Tutor Agent
Type 'quit' to stop.

Student: What is the column space of a matrix?

Tutor:


### Column space

The **column space** of a matrix is the set of all linear combinations of its columns.

If

$$
A = [\mathbf{a}_1\ \mathbf{a}_2\ \cdots\ \mathbf{a}_n],
$$

then

$$
\operatorname{Col}(A)
=
\operatorname{span}\{\mathbf{a}_1,\mathbf{a}_2,\ldots,\mathbf{a}_n\}.
$$

In other words, it is the smallest subspace containing all the columns of $A$. The columns may be independent, or some may be redundant combinations of others.

### Source

- Lecture 9 — Independence, basis, and dimension, 20:57–23:33
- [Watch this part](https://archive.org/download/MIT18.06S05_MP4/09.mp4#t=1257)


Student: In Lecture 21, how do you find the eigenvalues of a matrix?

Tutor:


### Finding eigenvalues in Lecture 21

For a matrix $A$, an eigenvalue $\lambda$ satisfies

$$
A\mathbf{x}=\lambda\mathbf{x}
$$

for some nonzero vector $\mathbf{x}$. Rearranging gives

$$
(A-\lambda I)\mathbf{x}=0.
$$

Because $\mathbf{x}\neq \mathbf{0}$, the matrix $A-\lambda I$ must be singular. Therefore,

$$
\det(A-\lambda I)=0.
$$

This equation is called the **characteristic equation**. To find the eigenvalues:

1. Form $A-\lambda I$ by subtracting $\lambda$ from each diagonal entry.
2. Compute $\det(A-\lambda I)$.
3. Set the determinant equal to zero.
4. Solve for $\lambda$.

For a triangular matrix, the eigenvalues can be read directly from the diagonal, including repeated eigenvalues.

### Source

- Lecture 21 — Eigenvalues and eigenvectors, 17:57–20:02
- [Watch this part](https://archive.org/download/MIT18.06S05_MP4/21.mp4#t=1077)
- Lecture 21 — Eigenvalues and eigenvectors, 46:27–48:16
- [Watch this part](https://archive.org/download/MIT18.06S05_MP4/21.mp4#t=2787)


Student: What is the difference between projection and least squares?

Tutor:


...


Student: Who won the World Cup?

Tutor:


That question is outside the available MIT 18.06 Linear Algebra course material.


Student: quir

Tutor:


Could you clarify what you mean by “quir”?


Student: quit

Demo ended.


# Step 8: Inspect and Export Weak Topics


In [ ]:
snapshot = agent.get_state(config)

weak_topics_raw = snapshot.values.get(
    "weak_topics",
    []
)

unique = {}
for item in weak_topics_raw:
    key = (
        item.get("topic"),
        item.get("lecture_id"),
    )
    unique[key] = item

weak_topics = list(unique.values())

print("Weak topics:")
for item in weak_topics:
    print(
        f"- {item['topic']} | "
        f"Lecture {item.get('lecture_number')} | "
        f"{item.get('timestamp')}"
    )


Weak topics:


In [ ]:
export_payload = {
    "thread_id": THREAD_ID,
    "lecture_start": rag_config["lecture_start"],
    "lecture_end": rag_config["lecture_end"],
    "lecture_numbers": rag_config["lecture_numbers"],
    "lecture_ids": rag_config["lecture_ids"],
    "weak_topics": weak_topics,
}

with open(
    WEAK_TOPICS_EXPORT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        export_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Exported:", WEAK_TOPICS_EXPORT)


Exported: /content/drive/MyDrive/AI_Engineering_Final_Project/outputs/all_lecture_weak_topics.json


## Notebook 3 Output

Notebook 3 now uses the RAG resources created in Notebook 2 instead of rebuilding the corpus.

Final separation:

```text
Notebook 2
STT
→ chunking
→ embeddings
→ persistent Chroma
→ retrieval
→ reranking
→ RAG

Notebook 3

Student Request
      ↓
Agent decides what the student wants
      ↓
┌──────────────────────────────────────────┐
│ Lecture question → search_lecture        │
│ Quiz request     → create_quiz           │
│ Quiz answer      → grade_quiz_answer     │
│ Weakness question → get_weak_topics      │
└──────────────────────────────────────────┘
      ↓
Memory / state carried across turns

What the Agent Can Do
- Lecture question: searches the lecture using the reranked RAG pipeline from Notebook 2.
- Quiz request: generates a multiple-choice question grounded in lecture content.
- Quiz answer: grades the answer deterministically and provides feedback.
- Weakness question: reads the student's recorded weak topics from agent state.
- Memory: InMemorySaver keeps the conversation, current quiz, and weak-topic state available across turns within the session.


In [ ]:
# Quick multi-lecture retrieval/source test

test_docs = retrieve_and_rerank(
    "What is the column picture?",
)

for doc in test_docs:
    start = doc.metadata["start_seconds"]

    print(
        f"Lecture {doc.metadata['lecture_number']} | "
        f"{doc.metadata['lecture_title']} | "
        f"{seconds_to_timestamp(start)} | "
        f"{build_timestamp_link(doc.metadata['video_url'], start)}"
    )


Lecture 1 | The geometry of linear equations | 20:56 | https://archive.org/download/MIT18.06S05_MP4/01.mp4#t=1256
Lecture 1 | The geometry of linear equations | 09:42 | https://archive.org/download/MIT18.06S05_MP4/01.mp4#t=582
Lecture 16 | Projection matrices and least squares | 33:45 | https://archive.org/download/MIT18.06S05_MP4/16.mp4#t=2025
